In [37]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_classic.retrievers import EnsembleRetriever

# For RAG (instead of RetrievalQA)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableMap

In [38]:
import os
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [39]:
file_name = "data_files/virtusa.pdf"
loader = PyPDFLoader(file_name)
docs = loader.load()
print(len(docs[0].page_content))

3806


In [40]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    add_start_index=True
)

chunks = text_splitter.split_documents(docs)

print(chunks[0].page_content)
print(chunks[0].metadata)
print(len(chunks))

Conﬁdenality Agreement 1.0
CONFIDENTIALITY AGREEMENT FOR INTERN
 
Conﬁdenality Agreement is entered between
Virtusa Consulng Services Private Limited
{'producer': '3-Heights™ PDF Security Shell 6.0.40.20304 (http://www.pdf-tools.com)', 'creator': 'PyPDF', 'creationdate': '2026-05-14T09:17:20+00:00', 'moddate': '2026-05-14T10:06:04+00:00', 'source': 'data_files/virtusa.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'start_index': 0}
31


In [41]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)

In [24]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

print("vector_db created.")

vector_db created.


In [42]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 2

In [43]:
document_ids = vectorstore.add_documents(documents=chunks)

print(document_ids[:3])

['3b02e5f2-e669-4b13-8e9c-e67e56cef353', 'bb5f5a92-4e5e-4703-b025-6e759180c57d', '41ff0a30-d5b7-49e6-93e0-21d76adb3a27']


In [44]:
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

In [45]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        semantic_retriever
    ],
    weights=[0.4, 0.6]
)

In [46]:
query = "What is Virtusa?"

results = hybrid_retriever.invoke(query)

for result in results:
    print(result.page_content)
    print("-" * 50)

associaon with Virtusa Consulng services private Limited.
 
 
Employee Signature
IP Address: 27.5.170.212
Name:Gokul Krishna Balaji Date of Signing:
14/05/2026
--------------------------------------------------
publicly available or a maer of public knowledge or public domain generally; (c) which is lawfully received by
the Recipient from a third party who is or was not bound in any conﬁdenal relaonship to the Discloser, or (d)
--------------------------------------------------
Conﬁdenality Agreement 1.0
CONFIDENTIALITY AGREEMENT FOR INTERN
 
Conﬁdenality Agreement is entered between
Virtusa Consulng Services Private Limited
--------------------------------------------------


In [47]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""

    retrieved_docs = hybrid_retriever.invoke(query)

    serialized = "\n\n".join(
        (
            f"Source: {doc.metadata}\n"
            f"Content: {doc.page_content}"
        )
        for doc in retrieved_docs
    )

    return serialized, retrieved_docs

In [48]:
model = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0.7,
    api_key=api_key,
    max_completion_tokens=1000
)

In [49]:
from pydantic import BaseModel

class ResponseFormat(BaseModel):
    message: str

In [50]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

tools = [retrieve_context]

prompt = """
You have access to a tool that retrieves context from a blog post. Use the tool to help answer user queries. Respond under 30 words.
"""

agent = create_agent(model=model, tools=tools, response_format=ToolStrategy(ResponseFormat), system_prompt=prompt)

In [51]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Virtusa?"}]}
)

In [52]:
result['structured_response'].message

'Virtusa is associated with Virtusa Consulting Services Private Limited, involved in confidentiality agreements and consulting services.'